## RAVEN ON MADELON CLUSTERING DATASET

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from scipy.sparse import issparse
from raven import raven 

Fetching OpenML dataset 1485 (Madelon)...
✅ Dataset loaded: 2600 samples, 500 features.

--- 1. Baseline (All Features) ---
Silhouette Score: 0.0083 (Higher is better)
ARI Score:        0.0280 (Match with true labels)

--- 2. Running RAVEN ---
Building redundancy graph...
Identifying essential and redundant features...
RAVEN selected 490 features (Removed 10)

--- 3. RAVEN Results ---
Silhouette Score: 0.0048
ARI Score:        0.0296

📈 Impact: ARI changed by 0.0016
🏆 Success: RAVEN removed noise and revealed the true clusters!


In [ ]:

DATASET_ID = 1485
print(f"Fetching OpenML dataset {DATASET_ID} ...")

dataset = fetch_openml(data_id=DATASET_ID, as_frame=True, parser='auto')
X_raw = dataset.data
y_true = dataset.target.astype(int) 

print(f" Dataset loaded: {X_raw.shape[0]} samples, {X_raw.shape[1]} features.")


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_dense_df = pd.DataFrame(X_scaled) 


In [ ]:

print("\nBaseline (All Features)")
kmeans_base = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)

sil_base = silhouette_score(X_scaled, labels_base)
ari_base = adjusted_rand_score(y_true, labels_base)

print(f"Silhouette Score: {sil_base:.4f}")
print(f"ARI Score: {ari_base:.4f} ")


In [ ]:

print("\n Running RAVEN")
essential_feats, redundant_feats = raven(
    data=X_dense_df,
    mode='df',
    sample_size=5000, 
    tau=0.9,
    
)

print(f"RAVEN selected {len(essential_feats)} features (Removed {len(redundant_feats)})")

X_raven = X_dense_df[essential_feats].to_numpy()

kmeans_raven = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_raven = kmeans_raven.fit_predict(X_raven)

sil_raven = silhouette_score(X_raven, labels_raven)
ari_raven = adjusted_rand_score(y_true, labels_raven)


In [ ]:

print(f"\n RAVEN Results")
print(f"Silhouette Score: {sil_raven:.4f}")
print(f"ARI Score: {ari_raven:.4f}")


Isolet DATASET

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.sparse import issparse
from raven import raven  

DATASET_ID = 300
print(f"Fetching OpenML dataset {DATASET_ID} (Isolet)...")


dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
X_raw = dataset.data
y_true = dataset.target
    
# Isolet labels are strings ('1', '2', ...). Convert to int for ARI score.
# Some versions have '1.0', so we handle that.
y_numeric = pd.factorize(y_true)[0]

print(f" Dataset loaded: {X_raw.shape[0]} samples, {X_raw.shape[1]} features.")
    
# Standard Scaling is crucial for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_dense_df = pd.DataFrame(X_scaled) 


Fetching OpenML dataset 300 (Isolet)...
 Dataset loaded: 7797 samples, 617 features.

--- 1. Baseline (All Features) ---
Silhouette Score: 0.0736
ARI Score:        0.4833

--- 2. Running RAVEN ---
Building redundancy graph...
Identifying essential and redundant features...
RAVEN selected 599 features.
Removed 18 redundant features.
Reduction: 2.9% of data removed.

--- RAVEN Results after removing redundant features ---
Silhouette Score: 0.0730
ARI Score:        0.5092

--- 🏁 Verdict ---
🏆 SUCCESS: RAVEN kept the clustering quality while using way less data.


In [ ]:

print("\nBaseline (All Features)")
# Isolet has 26 classes (A-Z)
kmeans_base = KMeans(n_clusters=26, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)
sil_base = silhouette_score(X_scaled, labels_base)
ari_base = adjusted_rand_score(y_numeric, labels_base)

print(f"Silhouette Score: {sil_base:.4f}")
print(f"ARI Score: {ari_base:.4f}")


In [ ]:

print("\n 2. Running RAVEN ")
    
    # We use a slightly lower tau (0.85) to be more aggressive with audio data
essential_feats, redundant_feats = raven(
        data=X_dense_df,
        mode='df',
        sample_size=8000, 
        tau=0.95
)

print(f"RAVEN selected {len(essential_feats)} features.")
print(f"Removed {len(redundant_feats)} redundant features.")
print(f"Reduction: {len(redundant_feats)/X_raw.shape[1]*100:.1f}% of data removed.")

 

In [ ]:
   # Select features
X_raven = X_dense_df[essential_feats].to_numpy()

    # Run K-Means on Reduced Data
kmeans_raven = KMeans(n_clusters=26, random_state=42, n_init=10)
labels_raven = kmeans_raven.fit_predict(X_raven)

sil_raven = silhouette_score(X_raven, labels_raven) 
ari_raven = adjusted_rand_score(y_numeric, labels_raven) 

print(f"\n RAVEN Results after removing redundant features ")
print(f"Silhouette Score: {sil_raven:.4f}")
print(f"ARI Score: {ari_raven:.4f}")


# RAVEN on CSV

In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold

from raven import raven

FILE_PATH = "datasets/kidney_gene.csv"
N_CLUSTERS = 2
RAVEN_SAMPLE = 100000
TAU = 0.6
RANDOM_STATE = 42



Raw CSV Shape: (8863, 624)
No label column detected (pure clustering)
Transposed matrix to Samples × Genes
Data Shape Used: (623, 8863) (Samples × Genes)
After variance filter: 8863 genes remain

--- Baseline Clustering (All Genes) ---
Silhouette Score: 0.2811

--- Running RAVEN ---
Building redundancy graph...
Identifying essential and redundant features...
RAVEN selected 4975 genes
Compression: 43.9%

--- Clustering on RAVEN Genes ---
Silhouette Score: 0.1950

--- SUMMARY ---
Baseline: 0.2811 silhouette using 8863 genes
RAVEN:    0.1950 silhouette using 4975 genes


In [ ]:
df = pd.read_csv(FILE_PATH)

print(f"Raw CSV Shape: {df.shape}")
label_col = None
for col in ["type", "class", "label", "phenotype"]:
    if col in df.columns:
        label_col = col
        break

if label_col:
    y_true = LabelEncoder().fit_transform(df[label_col].astype(str))
    df = df.drop(columns=[label_col])
    print(f"Detected label column: '{label_col}'")
else:
    y_true = None
    print("No label column detected (pure clustering)")

# Drop non-numeric columns defensively
X_df = df.select_dtypes(include=[np.number])

# If genes are rows → transpose
if X_df.shape[0] > X_df.shape[1]:
    X_df = X_df.T
    print("Transposed matrix to Samples x Genes")

print(f"Data Shape Used: {X_df.shape} (Samples x Genes)")



In [ ]:
selector = VarianceThreshold(threshold=0.0)
X_var = selector.fit_transform(X_df)

gene_names = X_df.columns[selector.get_support()]
X_var_df = pd.DataFrame(X_var, columns=gene_names, index=X_df.index)

print(f"After variance filter: {X_var_df.shape[1]} genes remain")


In [ ]:
X_log = np.log2(X_var_df + 1)

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_log),
    columns=X_log.columns,
    index=X_log.index
)



In [ ]:
print("\nBaseline Clustering (All Genes)")

kmeans_base = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)
labels_base = kmeans_base.fit_predict(X_scaled)

sil_base = silhouette_score(X_scaled, labels_base)
print(f"Silhouette Score: {sil_base:.4f}")

if y_true is not None:
    ari_base = adjusted_rand_score(y_true, labels_base)
    print(f"ARI Score: {ari_base:.4f}")



In [ ]:
print("\n Running RAVEN ")

essential_genes, redundant_genes = raven(
    data=X_scaled,
    mode="df",
    sample_size=RAVEN_SAMPLE,
    tau=TAU
)

print(f"RAVEN selected {len(essential_genes)} genes")
print(f"Compression: {100 * len(redundant_genes) / X_scaled.shape[1]:.1f}%")



In [ ]:
print("\nClustering on RAVEN Genes ")

X_raven = X_scaled[essential_genes]

kmeans_raven = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)
labels_raven = kmeans_raven.fit_predict(X_raven)

sil_raven = silhouette_score(X_raven, labels_raven)
print(f"Silhouette Score: {sil_raven:.4f}")

if y_true is not None:
    ari_raven = adjusted_rand_score(y_true, labels_raven)
    print(f"ARI Score: {ari_raven:.4f}")



In [ ]:

print("\n SUMMARY")
print(f"Baseline: {sil_base:.4f} silhouette using {X_scaled.shape[1]} genes")
print(f"RAVEN:    {sil_raven:.4f} silhouette using {len(essential_genes)} genes")


GENE EXPRESSION 

In [ ]:
DATASET_ID = 41084
RAVEN_SAMPLE = 1000   
TAU_THRESHOLD = 0.95 

print(f"Fetching OpenML dataset {DATASET_ID}...")

#Load Data (Robust Load)
try:
    # as_frame=False prevents memory crash on big datasets
    dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
    X_raw = dataset.data
    y_raw = dataset.target
    print(f"Type: {type(X_raw)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()


Fetching OpenML dataset 41084...
✅ Data loaded. Type: <class 'numpy.ndarray'>
   Preprocessing...
✅ Ready: 575 samples, 10304 features.
   Detected 20 classes in target.

--- 🟠 2. Running RAVEN ---
(Scanning 1000 samples...)


KeyboardInterrupt: 

In [ ]:

# Handle Sparse vs Dense
if issparse(X_raw):
    print(" Sparse Matrix detected. Converting to Dense...")
    X_dense = pd.DataFrame(X_raw.toarray())
else:
    # If dense, ensure we drop string columns
    X_temp = pd.DataFrame(X_raw)
    X_dense = X_temp.select_dtypes(include=[np.number])

if X_dense.shape[1] == 0:
    print("Error: No numeric data found!")
    exit()


In [ ]:

# Impute Missing Values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_dense)

# LOG TRANSFORM (Needs to be done for RNA or biological counts)
#   add +1 to avoid log(0) errors. 
# This handles the skewed nature of gene counts.

# X_log = np.log2(X_imputed - np.min(X_imputed) + 1)

#Standard Scaling (Required for K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Create Final DataFrame for RAVEN
# Use generic column names if headers are lost
feat_names = [f"gene_{i}" for i in range(X_scaled.shape[1])]
X_final_df = pd.DataFrame(X_scaled, columns=feat_names)

print(f"{X_final_df.shape[0]} samples, {X_final_df.shape[1]} features.")


In [ ]:

try:
    if y_raw is not None:
        le = LabelEncoder()
        y_numeric = le.fit_transform(y_raw)
        n_clusters = len(np.unique(y_numeric))
        has_labels = True
        print(f"Detected {n_clusters} classes in target.")
    else:
        raise ValueError
except:
    n_clusters = 2
    y_numeric = None
    has_labels = False
    print(" No valid labels found. Running Unsupervised (k=2).")


In [ ]:

print("\nBaseline (All Genes) ")
kmeans_base = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)

sil_base = silhouette_score(X_scaled, labels_base)
print(f"Silhouette Score: {sil_base:.4f}")

if has_labels:
    ari_base = adjusted_rand_score(y_numeric, labels_base)
    print(f"ARI Score: {ari_base:.4f} (Accuracy)")
else:
    ari_base = 0


In [ ]:

print("\nRunning RAVEN ")

essential_genes, redundant_genes = raven(
    data=X_final_df,
    mode='df',
    sample_size=RAVEN_SAMPLE, 
    tau=TAU_THRESHOLD
)

print(f"RAVEN selected {len(essential_genes)} genes.")
print(f"Dropped {len(redundant_genes)} redundant genes.")
print(f"Compression: {len(redundant_genes) / X_final_df.shape[1] * 100:.1f}%")


In [ ]:

print("\nClustering on RAVEN Genes")


X_raven = X_final_df[essential_genes].to_numpy()

kmeans_raven = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_raven = kmeans_raven.fit_predict(X_raven)
sil_raven = silhouette_score(X_raven, labels_raven)
print(f"Silhouette Score: {sil_raven:.4f}")

if has_labels:
    ari_raven = adjusted_rand_score(y_numeric, labels_raven)
    print(f"ARI Score: {ari_raven:.4f}")

  